In [3]:
import numpy as np
import cvxpy as cp

In [4]:
n = 3
p = 3

np.random.seed(1)
C = np.random.randn(n,n)

A = []
b = []

for i in range(p):
    A.append(np.random.randn(n,n))
    b.append(np.random.randn())

X = cp.Variable((n,n), symmetric = True)

constraints = [X >> 0]
constraints += [
    cp.trace(A[i]@X) == b[i] for i in range(p)
]

prob = cp.Problem(cp.Minimize(cp.trace(C @ X)),constraints)
prob.solve()

2.6543470585557287

In [5]:
print("The optimal value is", prob.value)
print("A solution X is")
print(X.value)

The optimal value is 2.6543470585557287
A solution X is
[[ 1.60805504 -0.59770125 -0.69575821]
 [-0.59770125  0.22228555  0.24689067]
 [-0.69575821  0.24689067  1.39679134]]


In [6]:
import numpy as np
import cvxpy as cp
import scipy.linalg as la

def partial_trace_A(rho, dim_A, dim_B):
    """
    B에 대한 Partial Trace를 수행하여 A의 상태를 반환 (Tr_B [rho])
    rho: (dim_A*dim_B, dim_A*dim_B)
    """
    rho_reshaped = rho.reshape((dim_A, dim_B, dim_A, dim_B))
    return np.einsum('jijk->ik', rho_reshaped)

def solve_fundamental_bound():
    """
    논문 Appendix D의 SDP (Eq. D6-D8) 구현
    Scenario: Appendix E의 Toy Example (Classical Correlation)
    """
    print("--- [Gemini] Fundamental Bound Calculation (Appendix D) ---")

    # 1. 시스템 설정 (Toy Example, Appendix E) [cite: 360, 361]
    # Qubit A + Qubit B
    d_A, d_B = 2, 2
    
    # Hamiltonians: H_x = epsilon * |1><1| (epsilon = 1.0)
    epsilon = 1.0
    H_local = np.array([[0, 0], [0, 1]], dtype=complex) * epsilon
    I = np.eye(2)
    
    H_A = np.kron(H_local, I) # H_A tensor I_B
    H_B = np.kron(I, H_local) # I_A tensor H_B
    H_AB = H_A + H_B          # Total Hamiltonian (Non-interacting)

    # 초기 상태: rho_AB = 0.5(|00><00| + |11><11|) [cite: 360]
    # 고전적으로 상관된 상태 (Classically Correlated)
    rho_AB_val = np.zeros((4, 4), dtype=complex)
    rho_AB_val[0, 0] = 0.5 # |00>
    rho_AB_val[3, 3] = 0.5 # |11>
    
    # 초기 에너지 및 엔트로피 계산
    rho_A_val = partial_trace_A(rho_AB_val, d_A, d_B)
    E_A_initial = np.real(np.trace(H_local @ rho_A_val))
    E_total_initial = np.real(np.trace(H_AB @ rho_AB_val))
    
    # von Neumann Entropy: S(rho) = -Tr(rho log rho) (log base e)
    # 0인 고유값 처리 주의
    evals = la.eigvalsh(rho_AB_val)
    evals = evals[evals > 1e-10]
    S_initial = -np.sum(evals * np.log(evals))

    print(f"Initial Energy E_A: {E_A_initial:.4f}")
    print(f"Total Energy E_AB:  {E_total_initial:.4f}")
    print(f"Initial Entropy S:  {S_initial:.4f}")

    # 2. SDP 변수 설정 (우리가 찾고자 하는 최적 상태 sigma_AB)
    sigma_AB = cp.Variable((4, 4), hermitian=True)

    # 3. 목적 함수: Minimize E(sigma_A) [cite: 351]
    # Tr_B[sigma_AB]를 변수로 표현하기 위해 Partial Trace 연산자를 행렬곱으로 구현하거나
    # 단순히 Total Hamiltonian H_A \otimes I_B 와의 내적으로 표현하면 됨.
    # E(sigma_A) = Tr((H_A \otimes I_B) * sigma_AB)
    objective = cp.Minimize(cp.real(cp.trace(H_A @ sigma_AB)))

    # 4. 제약 조건 (Appendix D)
    constraints = [
        sigma_AB >> 0,              # 양의 준정부호 (Positive Semidefinite)
        cp.trace(sigma_AB) == 1,    # Trace 1
        
        # Eq (D8): 에너지 보존 (Total Energy Conservation) [cite: 353]
        cp.real(cp.trace(H_AB @ sigma_AB)) == E_total_initial,
        
        # Eq (D7): 엔트로피 증가 (S(sigma) >= S(rho)) [cite: 352]
        # CVXPY의 von_neumann_entr는 concave 함수이므로, >= 제약조건은 convex set을 형성함.
        # 주의: cvxpy는 Base e를 사용
        cp.von_neumann_entr(sigma_AB) >= S_initial
    ]

    # 5. 최적화 수행
    problem = cp.Problem(objective, constraints)
    
    # SCS나 MOSEK 솔버 추천 (정확도 문제)
    try:
        problem.solve(solver=cp.SCS, verbose=False)
    except:
        print("SCS solver failed, trying default solver...")
        problem.solve()

    # 6. 결과 분석
    if sigma_AB.value is None:
        print("Optimization failed.")
        return

    sigma_AB_opt = sigma_AB.value
    E_A_final = np.real(np.trace(H_A @ sigma_AB_opt))
    Delta_E_A = E_A_final - E_A_initial
    
    print("\n--- Optimization Results ---")
    print(f"Final Energy E_A:   {E_A_final:.4f}")
    print(f"Delta E_A (Calculated): {Delta_E_A:.4f} epsilon")
    print(f"Delta E_A (Theory, Appx E): -0.2500 epsilon") # 
    
    # 결과 정합성 체크
    if np.isclose(Delta_E_A, -0.25, atol=1e-3):
        print("\n[Verification] SUCCESS: 논문의 이론적 예측값(-0.25)과 일치합니다.")
    else:
        print("\n[Verification] WARNING: 이론값과 차이가 있습니다. Solver 정밀도를 확인하세요.")

if __name__ == "__main__":
    solve_fundamental_bound()

--- [Gemini] Fundamental Bound Calculation (Appendix D) ---
Initial Energy E_A: 0.5000
Total Energy E_AB:  1.0000
Initial Entropy S:  0.6931

--- Optimization Results ---
Final Energy E_A:   0.1100
Delta E_A (Calculated): -0.3900 epsilon
Delta E_A (Theory, Appx E): -0.2500 epsilon

[Verification] WARNING: 이론값과 차이가 있습니다. Solver 정밀도를 확인하세요.
